# 第2回 演習: テキスト処理と文章生成

スライド: [第2回](../02-text/)

課題1〜3 で「次の語を引く機械」を作り、課題4 で同じ骨格をスタック電卓に差し替える。

## 進め方

- 先に準備セルを実行する
- `TODO` を埋めて、直下の確認セルを実行する
- 確認セル (`# [CHECK]`) は自分用。消しても教員の採点は別途走るが、残すこと
- 終わったら出力を残したまま保存し、kibaco から提出する

| 課題 | 配点 |
|---|---|
| 課題1 トークン化 | 20 |
| 課題2 次に来た語の表 | 15 |
| 課題2-b 続きやすい語 | 10 |
| 課題3 マルコフ連鎖で生成 | 20 |
| 課題4 逆ポーランド電卓 | 25 |
| 発展1 温度 | 5 |
| 発展2 2語を見る | 5 |


In [ ]:
# 準備 (最初に実行)
import urllib.request, pathlib

SAMPLE = '''I like cats. I like dogs. I love cats.
The cat sat on the mat. The dog sat on the log.

# コメント行 (課題1 で無視する)
A quick brown fox jumps over the lazy dog!
The quick brown dog meets a lazy fox.
'''
with open('sample.txt', 'w', encoding='utf-8') as f:
    f.write(SAMPLE)

# 長めの英文 (取れなくても先へ進める)
BOOK = 'alice.txt'
if pathlib.Path(BOOK).exists():
    print('長文テキストは手元にあります:', BOOK)
else:
    try:
        urllib.request.urlretrieve(
            'https://www.gutenberg.org/files/11/11-0.txt', BOOK)
        print('長文テキストを取得しました:', BOOK)
    except Exception as e:
        with open(BOOK, 'w', encoding='utf-8') as f:
            f.write(SAMPLE * 50)
        print('取得できなかったのでサンプルで代用します:', e)

print('準備完了')


In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認 (check / score) の準備。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '02'


---
## 課題1: トークン化 (20点)

ファイルを読み、**小文字の単語のリスト**を返す `tokenize_file(path)` を書け。

1. `#` で始まる行は無視する
2. 小文字にそろえる
3. 英字とアポストロフィだけを単語として取り出す (`.` `,` `!` などは捨てる)

**ヒント**: `re.findall(r"[a-z']+", line)` で、記号を除いた単語だけが取れる。
`line.split()` だと `'dog!'` のように記号がくっついたままになる。

### 例題

次のセルを実行して、動きを見る。

行を1つずつ処理する形。まず、部品を確かめる。


In [ ]:
# 例題
import re

line = "The dog's bone!  A cat."
print(line.startswith('#'))              # False (コメント行の判定)
print(re.findall(r"[a-z']+", line.lower()))
# ["the", "dog's", "bone", 'a', 'cat']


ファイルを回す形はこれ。


In [ ]:
# 例題
words = []
with open('sample.txt', encoding='utf-8') as f:
    for line in f:
        if line.startswith('#'):
            continue                     # コメント行は飛ばす
        words.extend(re.findall(r'[a-z]+', line.lower()))
print(words[:5])


`extend` はリストを**つなげる**。`append` だとリストの中にリストが入ってしまう。


In [ ]:
import re


def tokenize_file(path):
    tokens = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            # TODO
            pass
    return tokens


toks = tokenize_file('sample.txt')
print(len(toks), toks[:12])


In [ ]:
# [CHECK] q1
need('tokenize_file', arity=1)

toks = tokenize_file('sample.txt')
assert isinstance(toks, list)
assert toks[:6] == ['i', 'like', 'cats', 'i', 'like', 'dogs'], toks[:6]
assert 'これはコメント行' not in ''.join(toks), 'コメント行を無視すること'
assert 'dog!' not in toks and 'dog' in toks, '記号を落とすこと'
assert 'cats.' not in toks
assert all(t == t.lower() for t in toks), '小文字にそろえること'
assert len(toks) == 38, f'{len(toks)} 語。区切り方を見直すこと'
assert toks[-1] == 'fox', toks[-1]
print('OK')


---
## 課題2: 「次に来た語」の表 (15点)

トークン列から `{単語: Counter(次に来た単語)}` を作る `build_table(tokens)` を書け。

**ヒント**: `zip(tokens, tokens[1:])` で隣り合うペアが取れる。
`defaultdict(Counter)` を使うと、無いキーの初期化を書かずに済む。

### 例題

次のセルを実行して、動きを見る。

隣り合うペアを作って、数える。


In [ ]:
# 例題
from collections import defaultdict, Counter

toks = ['i', 'like', 'cats', 'i', 'like', 'dogs']
print(list(zip(toks, toks[1:])))
# [('i','like'), ('like','cats'), ('cats','i'), ('i','like'), ('like','dogs')]

table = defaultdict(Counter)
table['i']['like'] += 1                  # 無いキーでも書ける
table['i']['like'] += 1
print(dict(table['i']))                  # {'like': 2}


あとは、ペアを1つずつ回して `table[前][後] += 1` するだけ。


In [ ]:
from collections import defaultdict, Counter


def build_table(tokens):
    table = defaultdict(Counter)
    # TODO
    return table


table = build_table(tokenize_file('sample.txt'))
for w in ['i', 'the', 'like']:
    print(f'{w:6} -> {dict(table[w])}')


In [ ]:
# [CHECK] q2
need('build_table', arity=1)

table = build_table(tokenize_file('sample.txt'))
assert dict(table['i']) == {'like': 2, 'love': 1}, dict(table['i'])
assert dict(table['like']) == {'cats': 1, 'dogs': 1}
assert table['the']['quick'] == 1
assert sum(table['the'].values()) == 6, 'the の次に来た語は全部で6回'
assert len(build_table([])) == 0
assert len(build_table(['a'])) == 0, '1語だけならペアは作れない'
assert dict(build_table(['a', 'b', 'a', 'b'])['a']) == {'b': 2}

# 語をいくつ並べても、表に入る回数の合計は (語数 - 1) になるはず
holds('ペアの総数は語数-1',
      lambda ws: sum(sum(c.values()) for c in build_table(ws).values())
                 == max(len(ws) - 1, 0),
      [words(n) for n in range(0, 12)])
print('OK')


### 課題2-b: 続きやすい語 (10点)

表から「ある語の次に来やすい語」の上位 `n` 件を返す
`top_following(table, word, n)` を書け。

- 返り値は `[(単語, 回数), ...]` のリスト
- 回数の多い順。**回数が同じなら単語の辞書順**
- その語が表に無ければ空リスト

**ヒント**: `sorted(..., key=lambda kv: (-kv[1], kv[0]))`。
並び順を決めておかないと、同数の語の順序が実行のたびに変わってしまう。

### 例題

次のセルを実行して、動きを見る。

回数の多い順、同数なら辞書順に並べる。


In [ ]:
# 例題
from collections import Counter

c = Counter({'cat': 3, 'zoo': 1, 'ant': 1})
print(sorted(c.items(), key=lambda kv: (-kv[1], kv[0])))
# [('cat', 3), ('ant', 1), ('zoo', 1)]


`key` のタプルは「まず回数の符号を反転して昇順 (= 多い順)、次に単語の昇順」。
先頭 `n` 件を切り出すのは `[:n]`。表に無い語は `word not in table` で先に返す。


In [ ]:
def top_following(table, word, n=5):
    # TODO
    pass


big_tokens = tokenize_file(BOOK)
big_table = build_table(big_tokens)
print(f'{len(big_tokens):,} 語 / 異なり語数 {len(set(big_tokens)):,}')
print('the の次:', top_following(big_table, 'the', 5))


In [ ]:
# [CHECK] q2b
need('top_following', arity=2)

t = build_table(tokenize_file('sample.txt'))
assert top_following(t, 'i', 5) == [('like', 2), ('love', 1)]
assert top_following(t, 'like', 5) == [('cats', 1), ('dogs', 1)], '同数なら辞書順'
assert top_following(t, 'i', 1) == [('like', 2)]
assert top_following(t, 'zzz', 5) == [], '無い語なら空リスト'

r = top_following(big_table, 'the', 5)
assert len(r) == 5 and all(isinstance(w, str) and isinstance(c, int) for w, c in r)
assert [c for _w, c in r] == sorted([c for _w, c in r], reverse=True), '多い順に'
print('OK')


---
## 課題3: マルコフ連鎖で文章を生成 (20点)

2つの関数を書け。

- `pick_next(counter)`: 回数を重みにして、次の語を1つ選ぶ
- `generate(table, start, n)`: `start` から始めて**最大 `n` 語**つなげ、
  空白区切りの文字列を返す。次が無い語に来たらそこで終わる

**ヒント**: `random.choices(候補のリスト, weights=重みのリスト)` は**リストを返す**ので
`[0]` が要る。`random.choice` (単数形) は等確率なので**使わないこと**。

### 例題

次のセルを実行して、動きを見る。

重み付きのくじ引き。


In [ ]:
# 例題
import random
from collections import Counter

c = Counter({'like': 3, 'love': 1})
print(random.choices(list(c.keys()), weights=list(c.values()))[0])
# 3/4 の確率で 'like'


つなげる側の形。


In [ ]:
# 例題
word = 'the'
result = [word]
for _ in range(n - 1):
    if word not in table or not table[word]:
        break                            # 次が無ければ終わり
    word = pick_next(table[word])
    result.append(word)
print(' '.join(result))


In [ ]:
import random


def pick_next(counter):
    # TODO
    pass


def generate(table, start, n=30):
    word = start
    result = [word]
    # TODO
    return ' '.join(result)


random.seed(0)
for _ in range(3):
    print(generate(big_table, 'the', 25))
    print()


In [ ]:
# [CHECK] q3
need('pick_next', arity=1)
need('generate', arity=2)

from collections import Counter as _C
c = _C({'like': 2, 'love': 1})
assert set(pick_next(c) for _ in range(200)) == {'like', 'love'}, '両方選ばれること'
picks = [pick_next(c) for _ in range(600)]
assert 300 < picks.count('like') < 500, f"回数を重みにすること (like が {picks.count('like')}/600)"
assert pick_next(_C({'only': 5})) == 'only'

small = build_table(tokenize_file('sample.txt'))
out = generate(small, 'i', 10)
assert isinstance(out, str) and out.startswith('i '), out
assert len(out.split()) <= 11, '開始語 + 最大 n 語'
assert generate(small, 'i', 0) == 'i'

# 次が無い語で止まること
dead = build_table(['a', 'b'])
assert generate(dead, 'b', 10) == 'b'
assert generate(dead, 'a', 10) == 'a b'

# 表に沿った語しか出ないこと
chain = build_table(['x', 'y', 'x', 'y', 'x'])
for _ in range(20):
    ws = generate(chain, 'x', 6).split()
    assert all(a != b for a, b in zip(ws, ws[1:])), 'x と y が交互になるはず'
# 生成しても表そのものは壊さないこと
unchanged('generate', small, 'i', 5)
note(f'sample.txt の語 {len(small)} 種類から文を作れている')

print('OK')


---
## 課題4: ワード表方式の逆ポーランド電卓 (25点)

第1回の `@command` と同じ形のデコレータで、ワードを辞書に登録する。
スタックはただの Python のリストでよい (`append` が push、`pop` が pop)。

- `word(name)`: 関数を `WORDS[name]` に登録するデコレータを返す。
  登録される関数は**スタック1つだけを引数に取る**
- `evaluate(stack, text)`: 空白で区切り、`WORDS` にあれば呼び、
  無ければ整数として push。整数でもなければ `ValueError`

登録するワード:

| ワード | 動作 | スタックの変化 |
|---|---|---|
| `+` `-` `*` `/` | 2つ pop して計算し push | (a b → 結果) |
| `DUP` | 一番上を複製 | (a → a a) |
| `DROP` | 一番上を捨てる | (a → ) |
| `SWAP` | 上2つを入れ替え | (a b → b a) |
| `OVER` | 2番目を上にコピー | (a b → a b a) |
| `ROT` | 3番目を一番上へ | (a b c → b c a) |

**ヒント**: 引き算・割り算は**順序に注意**。先に pop されるのは右側の値。
`token.lstrip('-').isdigit()` で負数も含めて数値か判定できる。
割り算は整数除算 `//` でよい。

### 例題

次のセルを実行して、動きを見る。

ワードを1つ登録する形。`+` は書いてあるので、それ以外を同じ形で足す。


In [ ]:
# 例題
@word('NEG')                             # 課題には無い、練習用のワード
def _(s):
    s.append(-s.pop())                   # 上を1つ取って、符号を変えて戻す


s = [3]
WORDS['NEG'](s)
print(s)                                 # [-3]


`evaluate` の骨組み。数だけを扱う版から始めるとよい。


In [ ]:
# 例題
def evaluate(stack, text):
    for token in text.split():
        if token in WORDS:
            WORDS[token](stack)
        elif token.lstrip('-').isdigit():
            stack.append(int(token))
        else:
            raise ValueError(f'知らないワード: {token}')


引き算の順序に注意。`b, a = s.pop(), s.pop()` と書くと、
**先に取れる `b` が右側**。`3 4 -` は `3 - 4`。


In [ ]:
WORDS = {}


def word(name):
    def deco(func):
        # TODO: 登録して func をそのまま返す
        pass
    return deco


@word('+')
def _(s):
    b, a = s.pop(), s.pop()
    s.append(a + b)


# TODO: - * / DUP DROP SWAP OVER ROT を登録する


def evaluate(stack, text):
    # TODO
    pass


def calc_rpn(text):
    s = []
    evaluate(s, text)
    return s


print(calc_rpn('3 4 + 2 *'))


In [ ]:
# [CHECK] q4
need('WORDS', kind=dict)
need('word', arity=1)
need('evaluate', arity=2)

assert calc_rpn('3 4 +') == [7]
assert calc_rpn('10 3 -') == [7]        # 順序に注意
assert calc_rpn('3 4 + 2 *') == [14]
assert calc_rpn('2 3 4 * +') == [14]
assert calc_rpn('20 4 /') == [5]
assert calc_rpn('-5 3 +') == [-2]
assert calc_rpn('1 2 3') == [1, 2, 3]
assert calc_rpn('5 DUP *') == [25]
assert calc_rpn('1 2 DROP') == [1]
assert calc_rpn('1 2 SWAP') == [2, 1]
assert calc_rpn('1 2 OVER') == [1, 2, 1]
assert calc_rpn('1 2 3 ROT') == [2, 3, 1]
assert calc_rpn('1 2 3 ROT DROP SWAP') == [3, 2]

# 性質で確かめる: 数を積むだけなら、その順に並ぶ
holds('数を並べたらそのまま積まれる',
      lambda a, b: calc_rpn(f'{a} {b}') == [a, b], ints(2, lo=0, hi=99))
holds('SWAP を2回かけると元に戻る',
      lambda a, b: calc_rpn(f'{a} {b} SWAP SWAP') == [a, b], ints(2, lo=0, hi=99))
holds('足してから引くと元の数',
      lambda a, b: calc_rpn(f'{a} {b} + {b} -') == [a], ints(2, lo=0, hi=99))

# word() はデコレータとして働き、関数を壊さないこと
@word('NEG')
def _neg(s):
    s.append(-s.pop())
assert calc_rpn('5 NEG') == [-5], '実行中にワードを増やせること'

try:
    calc_rpn('3 FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')
print('OK')


---
# 発展課題

余力があれば。


## 発展1: 温度 (5点)

`pick_next_t(counter, temperature)` を書け。
重みを `回数 ** (1 / temperature)` にしてから選ぶ。

**ヒント**: 温度が小さいと差が拡大して最頻語ばかりに、
大きいと差が縮んでほぼ等確率になる。
確率から1つ選ぶ仕組みには、たいていこのつまみが付いてくる。

### 例題

次のセルを実行して、動きを見る。

重みを作り替えてから引くだけ。


In [ ]:
# 例題
counts = [3, 1]
t = 0.5
print([c ** (1 / t) for c in counts])     # [9.0, 1.0]  差が広がる
t = 3.0
print([c ** (1 / t) for c in counts])     # [1.44, 1.0] 差が縮む


`generate_t` は課題3 の `generate` と同じ形。呼ぶ相手を `pick_next_t` に変える。


In [ ]:
def pick_next_t(counter, temperature=1.0):
    # TODO
    pass


def generate_t(table, start, n=30, temperature=1.0):
    # TODO
    pass


random.seed(0)
for t in [0.2, 1.0, 3.0]:
    print(f'--- temperature = {t} ---')
    print(generate_t(big_table, 'the', 25, temperature=t))
    print()


In [ ]:
# [CHECK] b1
need('pick_next_t', arity=1)
need('generate_t', arity=2)

from collections import Counter as _C
c = _C({'a': 9, 'b': 1})
cold = [pick_next_t(c, 0.2) for _ in range(200)]
hot  = [pick_next_t(c, 5.0) for _ in range(200)]
assert cold.count('a') > 195, f'低温では最頻語に偏るはず (a が {cold.count("a")}/200)'
assert 60 < hot.count('a') < 175, f'高温では平坦に近づくはず (a が {hot.count("a")}/200)'
assert pick_next_t(_C({'only': 3}), 0.5) == 'only'

out = generate_t(big_table, 'the', 5, 1.0)
assert isinstance(out, str) and out.startswith('the'), out
assert len(out.split()) <= 6
print('OK')


## 発展2: 2語を見る (trigram) (5点)

直前**2語**をキーにした表を作る `build_table2(tokens)` と、
それを使う `generate2(table, start_pair, n)` を書け。

- `build_table2`: キーは `(a, b)` のタプル
- `generate2`: `start_pair` の2語から始め、最大 `n` 語つなげる

**ヒント**: `zip(tokens, tokens[1:], tokens[2:])` で3つ組が取れる。

### 例題

次のセルを実行して、動きを見る。

3つ組の取り方。


In [ ]:
# 例題
toks = ['a', 'b', 'c', 'd']
print(list(zip(toks, toks[1:], toks[2:])))
# [('a','b','c'), ('b','c','d')]


`table[(a, b)][c] += 1` と、キーをタプルにする。
生成側は直前2語を覚えておき、1語進むたびに `pair = (pair[1], next_word)` と更新。


In [ ]:
def build_table2(tokens):
    # TODO
    pass


def generate2(table, start_pair, n=30):
    # TODO
    pass


big_table2 = build_table2(big_tokens)
random.seed(0)
print('bigram :', generate(big_table, 'the', 25))
print()
print('trigram:', generate2(big_table2, ('the', 'queen'), 25))


In [ ]:
# [CHECK] b2
need('build_table2', arity=1)
need('generate2', arity=2)

t2 = build_table2(['a', 'b', 'c', 'a', 'b', 'd'])
assert dict(t2[('a', 'b')]) == {'c': 1, 'd': 1}, dict(t2[('a', 'b')])
assert dict(t2[('b', 'c')]) == {'a': 1}
assert len(build_table2(['a', 'b'])) == 0, '2語では3つ組は作れない'

out = generate2(t2, ('a', 'b'), 5)
assert out.startswith('a b '), out
assert len(out.split()) <= 7, '開始2語 + 最大 n 語'

# trigram のほうが組み合わせは多い
assert len(big_table2) > len(big_table), '組み合わせの数は増えるはず'

# 元の文に現れた並びしか出ないこと
src = ['p', 'q', 'r', 's', 'p', 'q', 'r', 's']
t3 = build_table2(src)
for _ in range(10):
    ws = generate2(t3, ('p', 'q'), 6).split()
    for a, b, c in zip(ws, ws[1:], ws[2:]):
        assert c in t3[(a, b)], f'{(a, b, c)} は元の文に無い並び'
print('OK')


---
## 自己確認

`score()` で全部まとめて確かめる。何度やってもよい。
`NG` の行に、次に何をすればいいかが出る。


In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

提出は kibaco から。`score()` と採点ページは下書き確認用。

1. 上から実行し、出力を残す
2. `ファイル → .ipynb をダウンロード`
3. 締切までに kibaco へ出す

確認セルが全部 `OK` なら提出してよい。確認セルを消しても教員の採点は別の検査で行う。
